In [1]:
import os
import subprocess

command = "source ~/.zprofile && env"
proc = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    shell=True,
    executable="/bin/zsh"
)

for line in proc.stdout:
    key, _, value = line.decode().partition("=")
    os.environ[key] = value.strip()

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-filesystem",
            "/Users/welcome"
        ]
    }
})

# Later in your async code
tools = await client.get_tools()

In [3]:
tools

[StructuredTool(name='read_file', description='Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.', args_schema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'path': {'type': 'string'}, 'tail': {'description': 'If provided, returns only the last N lines of the file', 'type': 'number'}, 'head': {'description': 'If provided, returns only the first N lines of the file', 'type': 'number'}}, 'required': ['path']}, metadata={'title': None, 'readOnlyHint': True, 'destructiveHint': None, 'idempotentHint': None, 'openWorldHint': None}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x1057344a0>),
 StructuredTool(name='read_text_file', description="Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the 

In [4]:
from langchain.agents import create_agent
agent = create_agent("openai:gpt-4.1", tools)
result = await agent.ainvoke({"messages": [{"role": "user", "content": "Find all files inside the /Users/welcome/Desktop/Tech_Learn_sai/ "}]})
print(f"Result: = {result}")

RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4.1-long-context in organization org-S55YIsHCoyDAJRPedgJfLj8Q on tokens per min (TPM): Limit 200000, Requested 2325331. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [5]:
from langchain.agents import create_agent
agent = create_agent("openai:gpt-4.1", tools)
result = await agent.ainvoke({"messages": [{"role": "user", "content": "List any 5 files in the root directory "}]})
print(f"Result: = {result}")

Result: = {'messages': [HumanMessage(content='List any 5 files in the root directory ', additional_kwargs={}, response_metadata={}, id='4b5ee72d-3c30-4535-a095-fe373959a355'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 1166, 'total_tokens': 1178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1024}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_1a2c4a5ede', 'id': 'chatcmpl-Czy16VmptTuTkVMNXCVAncwPa9eQR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bd9bf-3ef9-7f50-9fc1-ecc135845bdf-0', tool_calls=[{'name': 'list_allowed_directories', 'args': {}, 'id': 'call_3QQN2QFWETZGKVDMERAhXYsL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t

In [6]:
print("stream mode test:")
async for event in agent.astream({
    "messages": [{"role": "user", "content": "List any 5 files in the root directory"}]
}):
    print(event)

stream mode test:
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 1165, 'total_tokens': 1177, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_1a2c4a5ede', 'id': 'chatcmpl-Czy1svYWOL7ol4GTa4yn8tz95q0PB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bd9bf-f9e5-7793-aa5a-13fb2b02c8ff-0', tool_calls=[{'name': 'list_allowed_directories', 'args': {}, 'id': 'call_GEmB2Qwlmo1fOfCtkNyOOVHP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1165, 'output_tokens': 12, 'total_tokens': 1177, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'

In [8]:
print("stream mode test:")
async for event in agent.astream({
    "messages": [{"role": "user", "content": "List any 5 files in the root directory"}]
}):
    # Check if event has the structure you showed
    if isinstance(event, dict) and 'model' in event:
        model_data = event['model']
        if 'messages' in model_data and isinstance(model_data['messages'], list):
            for msg in model_data['messages']:
                # Check if this is an AIMessage-like object
                if hasattr(msg, 'content') and msg.content:
                    print(msg.content)
                # Alternative: if it's a dictionary representation
                elif isinstance(msg, dict) and 'content' in msg and msg['content']:
                    print(msg['content'])

stream mode test:
Here are 5 files in the root directory (/Users/welcome):

1. .CFUserTextEncoding
2. .DS_Store
3. .a2a_chat_history
4. .bash_profile
5. .claude.json

Let me know if you want more information or need files from any specific folder!
